# Bottle Inspection — YOLO11 Object Detection Pipeline

End-to-end **functional** notebook for detecting bottle defects using YOLO11-nano.  
All logic lives in pure functions — no classes, no global state.

## Label taxonomy (from domain specification)

| Category | Labels | Decision rule |
|----------|--------|---------------|
| **Always GOOD** | Embossing, Foam residue, No fault, Water drop | Always safe |
| **Conditionally FAULTY** | Air bubble, Chip, Contamination light, Glass imperfection, Scuffing, Scuffing heavy | FAULTY only if `bbox_area > threshold` |
| **Always FAULTY** | Break/Crack, Circlip, Contamination dark, Crown cap, Foil, Foreign object (×2), Glass shard, Insect, Label, Liquid, Mold, No base visible, Paint residue, Straw, Yeast residue | Always reject |

## Notebook sections

| # | Section |
|---|---------|
| 1 | Environment & dependencies |
| 2 | Configuration (single source of truth) |
| 3 | COCO parsing — ROI extraction + label→class mapping |
| 4 | COCO → YOLO label conversion |
| 5 | Dataset analysis & class distribution |
| 6 | Offline augmentation (minority-class oversampling) |
| 7 | Augmentation preview |
| 8 | Dataset YAML generation |
| 9 | Model setup |
| 10 | Training |
| 11 | Training curve visualisation |
| 12 | Validation & F1-optimised threshold sweep |
| 13 | Model export (ONNX / TensorRT) |
| 14 | Inference + conditional area-gate defect engine |
| 15 | Batch inspection & inspection report |
| 16 | Submission CSV generation |


## 1 · Environment & Dependencies

In [27]:
import subprocess, sys

def install_packages(packages: list[str]) -> None:
    """Install pip packages quietly."""
    for pkg in packages:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pkg],
            stdout=subprocess.DEVNULL,
        )

install_packages([
    'ultralytics>=8.3.0',
    'albumentations>=1.4.0',
    'pycocotools',
    'opencv-python-headless',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'onnx',
    'onnxruntime-gpu',
    'pandas',
    'tqdm',
    'Pillow',
    'pyyaml',
])

import json, os, random, shutil, warnings, math, time
from pathlib import Path
from typing import Any, Optional
from dataclasses import dataclass, field

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yaml
from PIL import Image
from tqdm import tqdm
from pycocotools.coco import COCO
from sklearn.metrics import (
    f1_score, precision_score, recall_score, confusion_matrix,
    precision_recall_curve,
)

import albumentations as A
import torch
from ultralytics import YOLO
import ultralytics

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

print(f'PyTorch     : {torch.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}',
      f'({torch.cuda.get_device_name(0)})'if torch.cuda.is_available() else '')
print(f'Ultralytics : {ultralytics.__version__}')

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
def check_gpus() -> str:
    """
    Detect available GPUs and return the device string for YOLO.
    Returns '0,1' if 2 GPUs found, '0' if only one, 'cpu' as fallback.
    """
    n = torch.cuda.device_count()
    print(f'GPUs available: {n}')
    for i in range(n):
        props = torch.cuda.get_device_properties(i)
        mem   = props.total_memory / 1024**3
        print(f'  GPU {i}: {props.name}  ({mem:.1f} GB)')

    if n >= 2:
        device_str = '0,1'
    elif n == 1:
        device_str = '0'
    else:
        device_str = 'cpu'

    print(f'Using device: {device_str}')
    return device_str


DEVICE_STR = check_gpus()

## 2 · Configuration — Single Source of Truth

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Edit ONLY this block. Every downstream function reads from CFG.
# ─────────────────────────────────────────────────────────────────────────────
CFG: dict[str, Any] = dict(

    # ── Paths ────────────────────────────────────────────────────────────────
    raw_images_dir   = Path('/kaggle/input/competitions/1st-krones-vision-ai-challenge/train_images'),
    test_images_dir  = Path('/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_images'),
    coco_json_train  = Path('/kaggle/input/competitions/1st-krones-vision-ai-challenge/train_annotations.json'),
    coco_json_test   = Path('/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_annotations_roi_only.json'),
    dataset_root     = Path('/kaggle/working/yolo_dataset'),
    output_dir       = Path('/kaggle/working/runs/bottle_detect'),
    model_export_dir = Path('/kaggle/working/models/exported'),
    predictions_csv  = Path('/kaggle/working/predictions.csv'),
    submission_csv   = Path('/kaggle/working/submission.csv'),

    # ── COCO ─────────────────────────────────────────────────────────────────
    # category_id=22 is the bottle-base ROI annotation
    roi_category_id  = 22,
    # Train/val/test split (of train annotations)
    val_fraction     = 0.15,
    test_fraction    = 0.10,

    # ── YOLO class names ─────────────────────────────────────────────────────
    # These are the FAULTY and CONDITIONAL defect names that become YOLO classes.
    # GOOD labels (water drop, foam residue, embossing, no fault) are EXCLUDED
    # from training targets — they are only used for the prediction logic.
    # Class index 0..N-1 maps exactly to these names in order.
    class_names = [
        # ── Always-FAULTY (0–15) ─────────────────────────────────────────────
        'break_crack',              # 0
        'circlip',                  # 1
        'contamination_dark',       # 2
        'crown_cap',                # 3
        'foil_semitransparent',     # 4
        'foreign_object_manual',    # 5
        'foreign_object_washing',   # 6
        'glass_shard',              # 7
        'insect',                   # 8
        'label',                    # 9
        'liquid',                   # 10
        'mold',                     # 11
        'no_base_visible',          # 12
        'paint_residue',            # 13
        'straw',                    # 14
        'yeast_residue',            # 15
        # ── Conditionally-FAULTY (16–21) ────────────────────────────────────
        'air_bubble',               # 16  — FAULTY if area > 500 px²
        'chip',                     # 17  — FAULTY if area > 200 px²
        'contamination_light',      # 18  — FAULTY if area > 180 px²
        'glass_imperfection',       # 19  — FAULTY if area > 100 px²
        'scuffing',                 # 20  — FAULTY if area > 75 000 px²
        'scuffing_heavy',           # 21  — FAULTY if area > 1 200 px²
    ],

    # ── Conditional area thresholds (pixel² in original image coordinates) ──
    # A detection of a conditional class is only a confirmed DEFECT if its
    # bounding-box area (w × h in px) exceeds this value.
    area_thresholds = {
        'air_bubble'          : 500,
        'chip'                : 200,
        'contamination_light' : 180,
        'glass_imperfection'  : 100,
        'scuffing'            : 75_000,
        'scuffing_heavy'      : 1_200,
    },

    # ── Always-GOOD labels — present in COCO but NOT trained as defects ──────
    # These are mapped and logged but skipped during label file generation.
    good_labels = {
        'water drop', 'water_drop',
        'foam residue', 'foam_residue',
        'embossing',
        'no fault', 'no_fault',
    },

    # ── COCO category name → CFG class_name mapping ──────────────────────────
    # Maps raw COCO category names (lowercase) to class_names above.
    # Add entries here if the COCO names differ from the class_names.
    coco_name_map = {
        'break / crack'                     : 'break_crack',
        'break/crack'                       : 'break_crack',
        'circlip'                           : 'circlip',
        'contamination dark'                : 'contamination_dark',
        'crown cap'                         : 'crown_cap',
        'foil / semitransparent'            : 'foil_semitransparent',
        'foil/semitransparent'              : 'foil_semitransparent',
        'foreign object - manual cleaning'  : 'foreign_object_manual',
        'foreign object - washing machine'  : 'foreign_object_washing',
        'glass shard'                       : 'glass_shard',
        'insect'                            : 'insect',
        'label'                             : 'label',
        'liquid'                            : 'liquid',
        'mold'                              : 'mold',
        'no base visible'                   : 'no_base_visible',
        'paint residue'                     : 'paint_residue',
        'straw'                             : 'straw',
        'yeast residue'                     : 'yeast_residue',
        'air bubble'                        : 'air_bubble',
        'chip'                              : 'chip',
        'contamination light'               : 'contamination_light',
        'glass imperfection'                : 'glass_imperfection',
        'scuffing'                          : 'scuffing',
        'scuffing heavy'                    : 'scuffing_heavy',
        # ROI label — skipped in label generation
        'roi'                               : '__roi__',
    },

    # ── Model ────────────────────────────────────────────────────────────────
    model_variant    = 'yolo26s.pt',   # YOLO11-nano; swap to 'yolo11s.pt' for more capacity
    img_size         = 640,            # 640

    # ── Training ─────────────────────────────────────────────────────────────
    epochs           = 150,
    patience         = 40,
    batch_size       = 32,
    lr0              = 1e-3,
    lrf              = 0.01,
    momentum         = 0.937,
    weight_decay     = 5e-4,
    warmup_epochs    = 5,
    cos_lr           = True,
    label_smoothing  = 0.05,
    # YOLO native augmentation params
    mosaic           = 1.0,
    mixup            = 0.15,
    copy_paste       = 0.3,
    degrees          = 15.0,
    translate        = 0.1,
    scale            = 0.5,
    shear            = 2.0,
    perspective      = 0.0005,
    flipud           = 0.1,
    fliplr           = 0.5,
    hsv_h            = 0.015,
    hsv_s            = 0.7,
    hsv_v            = 0.4,
    close_mosaic     = 15,
    workers          = 8,
    cache            = 'ram',    # 'ram' | 'disk' | False
    amp              = True,
    optimizer        = 'AdamW',

    # ── NMS / inference ──────────────────────────────────────────────────────
    conf_threshold   = 0.25,     # initial; refined by F1 sweep on val set
    iou_threshold    = 0.35,
    max_det          = 100,

    # ── Albumentations offline augmentation ──────────────────────────────────
    aug_fliplr       = 0.5,
    aug_flipud       = 0.1,
    aug_rotate       = 15.0,
    aug_brightness   = 0.10,
    aug_contrast     = 0.10,
    aug_blur_limit   = 3,
    aug_noise_var    = 15.0,
    clahe_clip       = 1.5,
    clahe_grid       = (8, 8),
    min_bbox_vis     = 0.25,     # drop augmented bbox if <25% visible

    # ── Offline oversampling ──────────────────────────────────────────────────
    # For classes with fewer annotations than median * multiplier, generate
    # synthetic copies via Albumentations
    oversample_multiplier = 3.0,

    
    # ── Per-class oversampling targets ────────────────────────────────────────
    # Derived from actual training run instance counts. Each value is the
    # minimum number of *additional* synthetic images to generate per class.
    # Classes with >500 annotations are left at their natural count.
    # Ultra-rare classes (insect, straw, circlip, paint_residue) get very high
    # multipliers because they currently contribute near-zero mAP.
    oversample_targets = {
        'break_crack'             : 400,   # 57  → 457
        'circlip'                 : 1200,  # 2   → 1202  ← near-zero, must oversample heavily
        'contamination_dark'      : 600,     # 1301 — already most common, skip
        'crown_cap'               : 0,     # 305  — mAP 0.976, fine
        'foil_semitransparent'    : 600,   # 8   → 608
        'foreign_object_manual'   : 0,     # 125  — mAP 0.891, fine
        'foreign_object_washing'  : 500,   # 82  → 582
        'glass_shard'             : 0,     # 305  — mAP 0.780, fine
        'insect'                  : 1500,  # 4   → 1204  ← only 4 instances!
        'label'                   : 400,   # 40  → 440
        'liquid'                  : 600,   # 17  → 617
        'mold'                    : 600,   # 784 → 1084
        'no_base_visible'         : 0,     # 564  — mAP 0.990, fine
        'paint_residue'           : 1200,   # 35  → 835   ← mAP 0.075, heavily underseen
        'straw'                   : 1200,   # 6   → 806   ← only 6 instances!
        'yeast_residue'           : 0,     # 904  — mAP 0.833, fine
        'air_bubble'              : 800,   # 518 → 918
        'chip'                    : 300,   # 103 → 403
        'contamination_light'     : 1000,   # 216 → 816   ← mAP 0.078, critical
        'glass_imperfection'      : 600,   # 25  → 625
        'scuffing'                : 400,     # 1513 — already large, keep natural
        'scuffing_heavy'          : 1500,   # 164 → 764   ← mAP 0.087, critical
    },

    # per_class_conf = {
    #     'contamination_dark'  : 0.10,  # lower threshold — catch more
    #     'contamination_light' : 0.08,  # very low — tiny defect
    #     'air_bubble'          : 0.08,
    #     'scuffing_heavy'      : 0.08,
    #     'scuffing'            : 0.10,
    # },
    # per_class_conf = {
    #     # Lower thresholds for high-FN classes (catch more)
    #     'mold'                : 0.12,
    #     'air_bubble'          : 0.08,
    #     'scuffing'            : 0.10,
    #     'contamination_light' : 0.08,
    #     'scuffing_heavy'      : 0.08,
    #     'glass_imperfection'  : 0.10,
    #     # Higher threshold for high-FP class (be more selective)
    #     'contamination_dark'  : 0.35,  # currently causing 40/92 FP
    #     'no_base_visible'     : 0.40,  # causing 4 FP — raise bar
    # },

    per_class_conf = {
          # Lower thresholds for high-FN classes (catch more)
          'mold'                : 0.12,
          'air_bubble'          : 0.08,
          'scuffing'            : 0.10,
          'contamination_light' : 0.08,
          'scuffing_heavy'      : 0.08,
          'glass_imperfection'  : 0.10,
          # Higher threshold for high-FP class (be more selective)
          'contamination_dark'  : 0.40,  # currently causing 40/92 FP
          'no_base_visible'     : 0.50,  # causing 4 FP — raise bar
          'yeast_residue'      : 0.45,
    },

    # ── Export ───────────────────────────────────────────────────────────────
    export_formats   = ['onnx'],
    half             = True,
)


def make_dirs(cfg: dict) -> None:
    """Create all output directories from CFG paths."""
    for key in ('dataset_root', 'output_dir', 'model_export_dir'):
        Path(cfg[key]).mkdir(parents=True, exist_ok=True)


make_dirs(CFG)
print(f"\n✅ Config loaded — {len(CFG['class_names'])} detection classes.")
print(f"   Always-FAULTY      : 16 classes (indices 0–15)")
print(f"   Conditionally-FAULTY: 6 classes (indices 16–21)")
print(f"   ROI category ID     : {CFG['roi_category_id']} (used for cropping only)")

## 3 · COCO Parsing — ROI Extraction + Label Mapping

In [ ]:
def _circle_from_bbox(
    bbox: list[float],
) -> tuple[int, int, int]:
    """Derive (cx, cy, radius) from a COCO [x, y, w, h] bounding box."""
    x, y, w, h = bbox
    return int(x + w / 2), int(y + h / 2), int(min(w, h) / 2)


def _circle_from_segmentation(
    segmentation: list,
) -> tuple[int, int, int]:
    """Fit the minimum enclosing circle to a COCO polygon segmentation."""
    pts = []
    for seg in segmentation:
        coords = np.array(seg, dtype=np.float32).reshape(-1, 2)
        pts.append(coords)
    pts_all = np.vstack(pts)
    (cx, cy), radius = cv2.minEnclosingCircle(pts_all.astype(np.float32))
    return int(cx), int(cy), int(radius)


def load_coco(
    coco_json: Path,
) -> tuple[dict, dict[int, str], dict[int, str]]:
    """
    Load a COCO JSON file and return:
      (coco_dict, id_to_filename, id_to_catname)
    """
    with open(coco_json, 'r') as f:
        coco = json.load(f)
    id_to_filename = {img['id']: img['file_name'] for img in coco['images']}
    id_to_catname  = {cat['id']: cat['name']      for cat in coco['categories']}
    print(f'  COCO categories ({len(id_to_catname)}):')
    for cid, cname in sorted(id_to_catname.items()):
        print(f'    id={cid:3d}  {cname}')
    return coco, id_to_filename, id_to_catname


def build_roi_map(
    coco:           dict,
    id_to_filename: dict[int, str],
    roi_category_id: int = 22,
) -> dict[str, tuple[int, int, int]]:
    """
    Build {basename: (cx, cy, radius)} from ROI annotations.
    Uses segmentation polygon when available, falls back to bbox.
    """
    roi_map: dict[str, tuple[int, int, int]] = {}
    skipped = 0
    for ann in coco.get('annotations', []):
        if ann.get('category_id') != roi_category_id:
            continue
        fname = Path(id_to_filename.get(ann['image_id'], '')).name
        if not fname:
            skipped += 1
            continue
        if ann.get('segmentation') and len(ann['segmentation']) > 0:
            try:
                cx, cy, r = _circle_from_segmentation(ann['segmentation'])
            except Exception:
                cx, cy, r = _circle_from_bbox(ann.get('bbox', [0,0,100,100]))
        elif ann.get('bbox') and len(ann['bbox']) == 4:
            cx, cy, r = _circle_from_bbox(ann['bbox'])
        else:
            skipped += 1
            continue
        roi_map[fname] = (cx, cy, r)

    print(f'  ROI map: {len(roi_map)} images mapped, {skipped} skipped.')
    return roi_map


def build_catid_to_classidx(
    id_to_catname: dict[int, str],
    class_names:   list[str],
    coco_name_map: dict[str, str],
    good_labels:   set[str],
) -> dict[int, int]:
    """
    Map COCO category_id → YOLO class index (0-based).
    Categories in good_labels or '__roi__' are mapped to None (skip).

    Returns {category_id: class_idx | None}.
    """
    name_to_idx = {n: i for i, n in enumerate(class_names)}
    mapping: dict[int, Optional[int]] = {}

    for cat_id, raw_name in id_to_catname.items():
        norm = raw_name.lower().strip()
        # 1. Direct match in class_names
        if norm in name_to_idx:
            mapping[cat_id] = name_to_idx[norm]
            continue
        # 2. Lookup via coco_name_map
        mapped = coco_name_map.get(norm)
        if mapped == '__roi__' or norm in good_labels:
            mapping[cat_id] = None     # skip
            continue
        if mapped and mapped in name_to_idx:
            mapping[cat_id] = name_to_idx[mapped]
            continue
        # 3. Check good_labels
        if norm in good_labels:
            mapping[cat_id] = None
            continue
        print(f'  ⚠ category "{raw_name}" (id={cat_id}) not mapped — add to coco_name_map.')
        mapping[cat_id] = None

    active = {k: v for k, v in mapping.items() if v is not None}
    skipped = {k: id_to_catname[k] for k, v in mapping.items() if v is None}
    print(f'  Class mapping: {len(active)} active, {len(skipped)} skipped.')
    print(f'  Skipped categories: {list(skipped.values())}')
    return mapping


# ── Load COCO ─────────────────────────────────────────────────────────────────
print('Loading training COCO annotations...')
coco, id_to_filename, id_to_catname = load_coco(CFG['coco_json_train'])

print('\nBuilding ROI map...')
roi_map = build_roi_map(coco, id_to_filename, CFG['roi_category_id'])

print('\nBuilding category → class index mapping...')
cat_map = build_catid_to_classidx(
    id_to_catname,
    CFG['class_names'],
    CFG['coco_name_map'],
    CFG['good_labels'],
)

## 4 · COCO → YOLO Label Conversion with ROI Cropping

In [ ]:
def crop_to_roi(
    img:    np.ndarray,
    cx:     int,
    cy:     int,
    radius: int,
    margin: float = 0.05,
) -> tuple[np.ndarray, int, int, int, int]:
    """
    Square-crop the image to the bottle-base ROI circle plus a margin.
    Returns (cropped_image, x_offset, y_offset, crop_w, crop_h).
    x_offset / y_offset are the pixel coordinates of the crop's top-left
    corner in the original image — needed to remap bboxes.
    """
    h, w = img.shape[:2]
    r_pad = int(radius * (1 + margin))
    x1    = max(0, cx - r_pad)
    y1    = max(0, cy - r_pad)
    x2    = min(w, cx + r_pad)
    y2    = min(h, cy + r_pad)
    return img[y1:y2, x1:x2], x1, y1, (x2 - x1), (y2 - y1)


def coco_bbox_to_yolo_in_crop(
    coco_bbox:  list[float],
    x_off:      int,
    y_off:      int,
    crop_w:     int,
    crop_h:     int,
) -> Optional[tuple[float, float, float, float]]:
    """
    Convert COCO [x_min, y_min, w, h] (absolute, original coords)
    to YOLO [cx, cy, w, h] (normalised, crop coords).
    Returns None if the bbox lies entirely outside the crop.
    """
    bx, by, bw, bh = coco_bbox

    # Clip to crop window
    x1 = max(bx,       x_off) - x_off
    y1 = max(by,       y_off) - y_off
    x2 = min(bx + bw,  x_off + crop_w) - x_off
    y2 = min(by + bh,  y_off + crop_h) - y_off

    if x2 <= x1 or y2 <= y1:
        return None   # box entirely outside crop

    # Normalise
    cx  = (x1 + x2) / 2 / crop_w
    cy  = (y1 + y2) / 2 / crop_h
    nw  = (x2 - x1) / crop_w
    nh  = (y2 - y1) / crop_h

    # Clamp
    cx  = min(max(cx,  0.0), 1.0)
    cy  = min(max(cy,  0.0), 1.0)
    nw  = min(max(nw,  1e-4), 1.0)
    nh  = min(max(nh,  1e-4), 1.0)

    return cx, cy, nw, nh


def split_image_ids(
    all_ids:       list[int],
    val_fraction:  float,
    test_fraction: float,
    seed:          int = 42,
) -> tuple[list[int], list[int], list[int]]:
    """Deterministic stratified split on image IDs."""
    rng = np.random.default_rng(seed)
    ids = list(all_ids)
    rng.shuffle(ids)
    n    = len(ids)
    n_val  = int(n * val_fraction)
    n_test = int(n * test_fraction)
    val_ids  = ids[:n_val]
    test_ids = ids[n_val: n_val + n_test]
    train_ids = ids[n_val + n_test:]
    return train_ids, val_ids, test_ids


def convert_coco_to_yolo(
    coco:           dict,
    id_to_filename: dict[int, str],
    cat_map:        dict[int, Optional[int]],
    roi_map:        dict[str, tuple[int, int, int]],
    raw_images_dir: Path,
    dataset_root:   Path,
    val_fraction:   float,
    test_fraction:  float,
) -> dict[str, dict]:
    """
    Full COCO → YOLO conversion:
      1. Split image IDs into train/val/test
      2. For each image: crop to ROI, save jpg, write YOLO label txt
      3. Skip GOOD-only images (no detectable defects) — YOLO treats
         images with no label file as background negatives automatically

    Returns conversion stats per split.
    """
    # Group annotations by image_id
    img_to_anns: dict[int, list] = {}
    for ann in coco.get('annotations', []):
        img_to_anns.setdefault(ann['image_id'], []).append(ann)

    all_ids = [img['id'] for img in coco['images']]
    train_ids, val_ids, test_ids = split_image_ids(
        all_ids, val_fraction, test_fraction,
    )
    split_map = {}
    for iid in train_ids: split_map[iid] = 'train'
    for iid in val_ids:   split_map[iid] = 'val'
    for iid in test_ids:  split_map[iid] = 'test'

    # Create dirs
    for split in ('train', 'val', 'test'):
        (dataset_root / split / 'images').mkdir(parents=True, exist_ok=True)
        (dataset_root / split / 'labels').mkdir(parents=True, exist_ok=True)

    stats = {s: {'images': 0, 'annotations': 0, 'skipped_img': 0,
                  'roi_fallback': 0}
             for s in ('train', 'val', 'test')}

    # Fixed fallback ROI for images without annotation
    FALLBACK_CX, FALLBACK_CY, FALLBACK_R = 640, 512, 480

    for img_info in tqdm(coco['images'], desc='Converting COCO → YOLO'):
        img_id   = img_info['id']
        split    = split_map.get(img_id, 'train')
        fname    = Path(img_info['file_name']).name
        stem     = Path(fname).stem

        # Find source image
        src = None
        for candidate in [
            raw_images_dir / fname,
            raw_images_dir / img_info['file_name'],
        ]:
            if Path(candidate).exists():
                src = Path(candidate)
                break
        if src is None:
            stats[split]['skipped_img'] += 1
            continue

        img = cv2.imread(str(src))
        if img is None:
            stats[split]['skipped_img'] += 1
            continue

        # ROI crop
        if fname in roi_map:
            cx, cy, r = roi_map[fname]
        else:
            cx, cy, r = FALLBACK_CX, FALLBACK_CY, FALLBACK_R
            stats[split]['roi_fallback'] += 1

        cropped, x_off, y_off, crop_w, crop_h = crop_to_roi(img, cx, cy, r)

        # Save cropped image
        dst_img = dataset_root / split / 'images' / f'{stem}.jpg'
        cv2.imwrite(str(dst_img), cropped)

        # Build YOLO label lines
        lines = []
        for ann in img_to_anns.get(img_id, []):
            cls_idx = cat_map.get(ann['category_id'])
            if cls_idx is None:      # GOOD label or ROI — skip
                continue
            bbox = ann.get('bbox')
            if not bbox or len(bbox) != 4:
                continue
            yolo_box = coco_bbox_to_yolo_in_crop(
                bbox, x_off, y_off, crop_w, crop_h,
            )
            if yolo_box is None:
                continue
            cx_n, cy_n, w_n, h_n = yolo_box
            lines.append(
                f'{float(cls_idx)} {cx_n:.6f} {cy_n:.6f} {w_n:.6f} {h_n:.6f}'
            )
            stats[split]['annotations'] += 1

        # Write label file (empty = background-only image)
        dst_lbl = dataset_root / split / 'labels' / f'{stem}.txt'
        dst_lbl.write_text('\n'.join(lines))
        stats[split]['images'] += 1

    for s, st in stats.items():
        print(f'  [{s}] images={st["images"]} | annotations={st["annotations"]} '
              f'| skipped={st["skipped_img"]} | roi_fallback={st["roi_fallback"]}')
    return stats


conv_stats = convert_coco_to_yolo(
    coco, id_to_filename, cat_map, roi_map,
    raw_images_dir = CFG['raw_images_dir'],
    dataset_root   = CFG['dataset_root'],
    val_fraction   = CFG['val_fraction'],
    test_fraction  = CFG['test_fraction'],
)
print('\n✅ Conversion complete.')

## 5 · Dataset Analysis & Class Distribution

In [ ]:
def count_class_instances(
    label_dir:   Path,
    num_classes: int,
) -> np.ndarray:
    """Count annotation instances per class from YOLO label txt files."""
    counts = np.zeros(num_classes, dtype=np.int64)
    for lbl in label_dir.glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                counts[int(float(parts[0]))] += 1
    return counts


def compute_class_weights(counts: np.ndarray) -> np.ndarray:
    """Inverse-frequency weights, normalised to sum to num_classes."""
    c = counts.astype(float)
    c = np.where(c == 0, 1, c)
    w = 1.0 / c
    return w / w.sum() * len(c)


def plot_class_distribution(
    counts:      np.ndarray,
    class_names: list[str],
    title:       str = 'Class Distribution — Training Set',
) -> None:
    colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(class_names)))
    fig, ax = plt.subplots(figsize=(11, 6))
    bars = ax.barh(class_names, counts, color=colors, edgecolor='white', linewidth=0.5)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_width() + counts.max() * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{cnt:,}', va='center', fontsize=8)
    ax.set_xlabel('Annotation count')
    ax.set_title(title, fontweight='bold')
    ax.axvline(np.median(counts[counts > 0]), color='navy',
               linestyle='--', linewidth=1, label='Median')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_bbox_stats(label_dir: Path, class_names: list[str]) -> None:
    """Histogram of normalised bbox widths, heights, and areas."""
    widths, heights, areas, classes = [], [], [], []
    for lbl in label_dir.glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            p = line.strip().split()
            if len(p) == 5:
                c, _, _, w, h = int(float(p[0])), *map(float, p[1:])
                widths.append(float(p[3]))
                heights.append(float(p[4]))
                areas.append(float(p[3]) * float(p[4]))
                classes.append(class_names[c] if c < len(class_names) else str(c))

    df = pd.DataFrame(dict(width=widths, height=heights, area=areas, cls=classes))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, col, color in zip(axes, ['width', 'height', 'area'],
                               ['steelblue', 'coral', 'mediumseagreen']):
        ax.hist(df[col], bins=60, color=color, edgecolor='white', linewidth=0.3)
        ax.set_title(f'BBox {col.capitalize()} (normalised)')
        ax.set_ylabel('Frequency')
    plt.suptitle('Bounding-Box Statistics — Training Set', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    print('\nMedian bbox stats per class:')
    print(df.groupby('cls')[['width', 'height', 'area']].median().round(4))


train_lbl_dir = CFG['dataset_root'] / 'train' / 'labels'
nc = len(CFG['class_names'])

class_counts  = count_class_instances(train_lbl_dir, nc)
class_weights = compute_class_weights(class_counts)

df_dist = pd.DataFrame({
    'class'  : CFG['class_names'],
    'count'  : class_counts,
    'weight' : np.round(class_weights, 4),
    'type'   : (['always_faulty'] * 16) + (['conditional'] * 6),
})
print(df_dist.to_string(index=False))

plot_class_distribution(class_counts, CFG['class_names'])
plot_bbox_stats(train_lbl_dir, CFG['class_names'])

## 6 · Offline Augmentation — Minority-Class Oversampling

In [ ]:
def build_train_augmentation(cfg: dict) -> A.Compose:
    """
    Heavy bbox-safe augmentation pipeline for offline oversampling.

    Design choices informed by the classification model analysis:
    - ColorJitter kept NARROW (0.10) — wider ranges create fake contamination_light
    - CoarseDropout REMOVED — creates false dark patches on good bottles
    - Sharpen ADDED — helps distinguish scuffing texture from smooth glass
    - RandomShadow ADDED — simulates conveyor-belt edge shadows
    - CLAHE clip kept LOW (1.5) — aggressive CLAHE makes water drops
      look similar to contamination_dark in the processed image
    """
    return A.Compose(
        [
            # ── Geometric ────────────────────────────────────────────────────
            A.HorizontalFlip(p=cfg['aug_fliplr']),
            A.VerticalFlip(p=cfg['aug_flipud']),
            A.Rotate(limit=int(cfg['aug_rotate']),
                     border_mode=cv2.BORDER_REFLECT_101, p=0.7),
            A.Perspective(scale=(0.02, 0.07), p=0.2),
            A.ShiftScaleRotate(
                shift_limit=0.06, scale_limit=0.15,
                rotate_limit=int(cfg['aug_rotate']),
                border_mode=cv2.BORDER_REFLECT_101, p=0.5,
            ),
            # ── Colour ───────────────────────────────────────────────────────
            A.RandomBrightnessContrast(
                brightness_limit=cfg['aug_brightness'],
                contrast_limit=cfg['aug_contrast'], p=0.6,
            ),
            A.HueSaturationValue(
                hue_shift_limit=5, sat_shift_limit=20,
                val_shift_limit=20, p=0.4,
            ),
            A.CLAHE(
                clip_limit=cfg['clahe_clip'],
                tile_grid_size=cfg['clahe_grid'], p=0.4,
            ),
            A.ImageCompression(quality_lower=75, quality_upper=100, p=0.3),
            # ── Texture / defect visibility ───────────────────────────────────
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.9, 1.1), p=0.35),
            # ── Blur / noise ─────────────────────────────────────────────────
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, cfg['aug_blur_limit'] * 2 + 1), p=1.0),
                A.MotionBlur(blur_limit=5, p=1.0),
            ], p=0.25),
            A.GaussNoise(var_limit=(5.0, cfg['aug_noise_var']), p=0.25),
            # ── Lighting simulation ───────────────────────────────────────────
            A.RandomShadow(num_shadows_lower=1, num_shadows_upper=2,
                           shadow_dimension=4, p=0.2),
            A.RandomFog(fog_coef_lower=0.03, fog_coef_upper=0.15, p=0.1),
            A.GridDistortion(num_steps=5, distort_limit=0.10, p=0.15),
        ],
        bbox_params=A.BboxParams(
            format='yolo',
            label_fields=['class_labels'],
            min_visibility=0.25,
        ),
    )


# def offline_oversample_minorities(
#     label_dir:   Path,
#     image_dir:   Path,
#     class_counts: np.ndarray,
#     class_names: list[str],
#     cfg:         dict,
#     target_mult: float = 3.0,
# ) -> int:
#     """
#     For each class whose count < median * target_mult, generate synthetic
#     augmented images until the deficit is filled.

#     Key: only images that CONTAIN the underrepresented class are used as
#     source images for that class's oversampling.
#     """
#     if not label_dir.exists() or not image_dir.exists():
#         print('⚠ Dirs not found — skipping offline oversampling.')
#         return 0

#     median_count = int(np.median(class_counts[class_counts > 0]))
#     target       = int(median_count * target_mult)
#     aug          = build_train_augmentation(cfg)
#     created      = 0

    # for cls_idx, cls_name in enumerate(class_names):
    #     current = int(class_counts[cls_idx])
    #     if current >= target:
    #         continue
    #     need = target - current
    #     print(f'  ↑ {cls_name:30s}: {current:5d} → target {target:5d} (+{need})')

    #     # Source files: label files containing this class
    #     src_files = [
    #         lbl for lbl in label_dir.glob('*.txt')
    #         if any(int(float(l.split()[0])) == cls_idx
    #                for l in lbl.read_text().strip().splitlines() if l.strip())
    #     ]
    #     if not src_files:
    #         print(f'    ⚠ No source images for {cls_name}.')
    #         continue

    #     for i in range(need):
    #         lbl_path = random.choice(src_files)
    #         stem = lbl_path.stem

    #         img_path = None
    #         for ext in ('.jpg', '.jpeg', '.png'):
    #             cand = image_dir / f'{stem}{ext}'
    #             if cand.exists():
    #                 img_path = cand
    #                 break
    #         if img_path is None:
    #             continue

    #         img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    #         bboxes, labels = [], []
    #         for line in lbl_path.read_text().strip().splitlines():
    #             parts = line.strip().split()
    #             if len(parts) == 5:
    #                 labels.append(int(float(parts[0])))
    #                 bboxes.append(list(map(float, parts[1:])))

    #         try:
    #             result = aug(image=img, bboxes=bboxes, class_labels=labels)
    #         except Exception:
    #             continue

    #         aug_img   = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
    #         new_stem  = f'{stem}_aug_{cls_name}_{i:05d}'

    #         cv2.imwrite(str(image_dir / f'{new_stem}.jpg'), aug_img)

    #         aug_lines = [
    #             f"{cl} {' '.join(f'{v:.6f}' for v in bb)}"
    #             for cl, bb in zip(result['class_labels'], result['bboxes'])
    #         ]
    #         (label_dir / f'{new_stem}.txt').write_text('\n'.join(aug_lines))
    #         created += 1

    # print(f'\n✅ Offline oversampling: {created} synthetic images created.')
    # return created


# n_synth = offline_oversample_minorities(
#     label_dir    = train_lbl_dir,
#     image_dir    = CFG['dataset_root'] / 'train' / 'images',
#     class_counts = class_counts,
#     class_names  = CFG['class_names'],
#     cfg          = CFG,
#     target_mult  = CFG['oversample_multiplier'],
# )


def oversample_class(
    cls_idx:   int,
    cls_name:  str,
    n_extra:   int,
    label_dir: Path,
    image_dir: Path,
    aug:       A.Compose,
) -> int:
    """
    Generate n_extra synthetic images containing cls_idx.
    Source images are sampled with replacement from the pool of images
    that already contain at least one annotation of this class.
    Returns the number of images actually created.
    """
    src_files = [
        lbl for lbl in label_dir.glob('*.txt')
        if any(int(float(l.split()[0])) == cls_idx
               for l in lbl.read_text().strip().splitlines() if l.strip())
    ]
    if not src_files:
        print(f'    ⚠ No source images for {cls_name} — skipping.')
        return 0

    created = 0
    for i in range(n_extra):
        lbl_path = random.choice(src_files)
        stem     = lbl_path.stem

        img_path = None
        for ext in ('.jpg', '.jpeg', '.png'):
            cand = image_dir / f'{stem}{ext}'
            if cand.exists():
                img_path = cand
                break
        if img_path is None:
            continue

        # img    = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        raw = cv2.imread(str(img_path))

        if raw is None:
            continue
        
        img = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        bboxes, labels = [], []
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                labels.append(int(float(parts[0])))
                bboxes.append(list(map(float, parts[1:])))

        try:
            result = aug(image=img, bboxes=bboxes, class_labels=labels)
        except Exception:
            continue

        # Only save if the target class survived augmentation
        if cls_idx not in result['class_labels']:
            continue

        aug_img  = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
        new_stem = f'{stem}_os_{cls_name}_{i:05d}'
        cv2.imwrite(str(image_dir / f'{new_stem}.jpg'), aug_img)

        aug_lines = [
            f"{cl} {' '.join(f'{v:.6f}' for v in bb)}"
            for cl, bb in zip(result['class_labels'], result['bboxes'])
        ]
        (label_dir / f'{new_stem}.txt').write_text('\n'.join(aug_lines))
        created += 1

    return created


def run_targeted_oversampling(
    label_dir:         Path,
    image_dir:         Path,
    class_names:       list[str],
    oversample_targets: dict[str, int],
    cfg:               dict,
) -> dict[str, int]:
    """
    Run per-class oversampling using the explicit targets in CFG.

    Why explicit targets instead of median-based auto-targets?
    ──────────────────────────────────────────────────────────
    The first training run revealed extreme imbalance — insect has 4 instances,
    contamination_dark has 1301. A median-based approach would set target at
    ~300, leaving insect at 304 but contamination_dark unchanged.
    Using explicit targets lets us be surgical:
      - Ultra-rare (circlip, insect, straw): target 1200 synthetic images
      - Poor-performing (contamination_light, scuffing_heavy): target 600
      - Good-performing (crown_cap, no_base_visible): skip entirely

    Returns dict of {class_name: n_created}.
    """
    if not label_dir.exists() or not image_dir.exists():
        print('⚠ Dataset dirs not found — skipping oversampling.')
        return {}

    aug     = build_train_augmentation(cfg)
    summary = {}

    for cls_idx, cls_name in enumerate(class_names):
        target = oversample_targets.get(cls_name, 0)
        if target <= 0:
            summary[cls_name] = 0
            continue

        print(f'  ↑ {cls_name:30s} +{target} synthetic images...')
        n = oversample_class(cls_idx, cls_name, target, label_dir, image_dir, aug)
        summary[cls_name] = n
        print(f'    → created {n}')

    total = sum(summary.values())
    print(f'\n✅ Oversampling complete — {total} total synthetic images created.')
    return summary


# ── Recount after oversampling to confirm ─────────────────────────────────────
def print_class_count_comparison(
    label_dir:   Path,
    class_names: list[str],
    before:      np.ndarray,
) -> None:
    """Print before/after annotation counts side-by-side."""
    after   = count_class_instances(label_dir, len(class_names))
    df = pd.DataFrame({
        'class'  : class_names,
        'before' : before,
        'after'  : after,
        'added'  : after - before,
    })
    df['ratio'] = (df['after'] / df['before'].replace(0, 1)).round(1)
    print(df.to_string(index=False))
    return after


# ── Run oversampling ──────────────────────────────────────────────────────────
before_counts = count_class_instances(
    CFG['dataset_root'] / 'train' / 'labels',
    len(CFG['class_names']),
)

os_summary = run_targeted_oversampling(
    label_dir          = CFG['dataset_root'] / 'train' / 'labels',
    image_dir          = CFG['dataset_root'] / 'train' / 'images',
    class_names        = CFG['class_names'],
    oversample_targets = CFG['oversample_targets'],
    cfg                = CFG,
)

print('\nBefore vs After oversampling:')
after_counts = print_class_count_comparison(
    CFG['dataset_root'] / 'train' / 'labels',
    CFG['class_names'],
    before_counts,
)


## 7 · Augmentation Preview

In [ ]:
def visualise_augmentation(
    image_dir:   Path,
    label_dir:   Path,
    class_names: list[str],
    cfg:         dict,
    n_samples:   int = 4,
) -> None:
    """Show original vs. augmented images with bounding boxes."""
    img_files = [f for f in list(image_dir.glob('*.jpg'))[:n_samples * 3]
                 if (label_dir / f'{f.stem}.txt').exists() and
                    (label_dir / f'{f.stem}.txt').read_text().strip()][:n_samples]

    if not img_files:
        print('⚠ No labelled images found for preview.')
        return

    aug      = build_train_augmentation(cfg)
    PALETTE  = plt.cm.tab20.colors

    def draw_boxes(ax, img_rgb, bboxes, labels, title=''):
        ax.imshow(img_rgb)
        h, w = img_rgb.shape[:2]
        for (cx, cy, bw, bh), lbl in zip(bboxes, labels):
            x1  = int((cx - bw / 2) * w)
            y1  = int((cy - bh / 2) * h)
            bwp = int(bw * w)
            bhp = int(bh * h)
            col = PALETTE[int(lbl) % len(PALETTE)]
            ax.add_patch(mpatches.Rectangle(
                (x1, y1), bwp, bhp, lw=2, edgecolor=col, facecolor='none'))
            ax.text(x1, y1 - 3, class_names[int(lbl)] if int(lbl) < len(class_names) else str(int(lbl)),
                    color='white', fontsize=6,
                    bbox=dict(facecolor=col, alpha=0.8, pad=1))
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    fig, axes = plt.subplots(len(img_files), 2,
                              figsize=(12, len(img_files) * 4))
    if len(img_files) == 1:
        axes = [axes]

    for i, img_path in enumerate(img_files):
        raw = cv2.imread(str(img_path))

        if raw is None:
            continue
        
        img = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        # img      = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        lbl_path = label_dir / f'{img_path.stem}.txt'
        bboxes, labels = [], []
        for line in lbl_path.read_text().strip().splitlines():
            p = line.strip().split()
            if len(p) == 5:
                labels.append(int(float(p[0])))
                bboxes.append(list(map(float, p[1:])))

        draw_boxes(axes[i][0], img, bboxes, labels, 'Original')

        # if len(result['bboxes']) == 0:
        #     continue

        result = aug(image=img, bboxes=bboxes, class_labels=labels)
        draw_boxes(axes[i][1], result['image'],
                   result['bboxes'], result['class_labels'], 'Augmented')

    plt.suptitle('Augmentation Preview', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


visualise_augmentation(
    image_dir   = CFG['dataset_root'] / 'train' / 'images',
    label_dir   = train_lbl_dir,
    class_names = CFG['class_names'],
    cfg         = CFG,
)

## 8 · Dataset YAML

In [ ]:
def write_dataset_yaml(
    dataset_root: Path,
    class_names:  list[str],
) -> Path:
    """Write YOLO dataset.yaml and return its path."""
    yaml_path = dataset_root / 'dataset.yaml'
    data = {
        'path' : str(dataset_root.resolve()),
        'train': 'train/images',
        'val'  : 'val/images',
        'test' : 'test/images',
        'nc'   : len(class_names),
        'names': class_names,
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False)
    print(f'✅ dataset.yaml → {yaml_path}')
    print(yaml.dump(data, default_flow_style=False, sort_keys=False))
    return yaml_path


yaml_path = write_dataset_yaml(CFG['dataset_root'], CFG['class_names'])

## 9 · Model Setup

In [ ]:
def load_yolo_model(cfg: dict) -> YOLO:
    """
    Load YOLO model. Tries the configured variant first,
    then falls back through YOLO11n → YOLO11s → YOLOv8n.
    """
    fallback_chain = [
        cfg['model_variant'],
        'yolo11n.pt',
        'yolo11s.pt',
        'yolov8n.pt',
    ]
    for variant in dict.fromkeys(fallback_chain):  # dedup, preserve order
        try:
            model = YOLO(variant)
            print(f'✅ Loaded: {variant}')
            return model
        except Exception as e:
            print(f'  ⚠ {variant}: {e}')
    raise RuntimeError('No YOLO variant could be loaded.')


def build_train_args(cfg: dict, yaml_path: Path) -> dict:
    """
    Assemble all kwargs for model.train().
    Augmentation values mirror the classification model's winning config.
    """
    return dict(
        data           = str(yaml_path),
        epochs         = cfg['epochs'],
        patience       = cfg['patience'],
        batch          = cfg['batch_size'],
        imgsz          = cfg['img_size'],
        lr0            = cfg['lr0'],
        lrf            = cfg['lrf'],
        momentum       = cfg['momentum'],
        weight_decay   = cfg['weight_decay'],
        warmup_epochs  = cfg['warmup_epochs'],
        cos_lr         = cfg['cos_lr'],
        label_smoothing= cfg['label_smoothing'],
        mosaic         = cfg['mosaic'],
        mixup          = cfg['mixup'],
        copy_paste     = cfg['copy_paste'],
        degrees        = cfg['degrees'],
        translate      = cfg['translate'],
        scale          = cfg['scale'],
        shear          = cfg['shear'],
        perspective    = cfg['perspective'],
        flipud         = cfg['flipud'],
        fliplr         = cfg['fliplr'],
        hsv_h          = cfg['hsv_h'],
        hsv_s          = cfg['hsv_s'],
        hsv_v          = cfg['hsv_v'],
        close_mosaic   = cfg['close_mosaic'],
        workers        = cfg['workers'],
        cache          = cfg['cache'],
        amp            = cfg['amp'],
        optimizer      = cfg['optimizer'],
        device         = '0,1',
        project        = str(cfg['output_dir']),
        name           = 'train',
        exist_ok       = True,
        plots          = True,
        verbose        = True,
    )


model    = load_yolo_model(CFG)
train_args = build_train_args(CFG, yaml_path)

print('\nTraining arguments:')
for k, v in train_args.items():
    print(f'  {k:20s}: {v}')

## 10 · Training

In [ ]:
# ckpt = torch.load('best.pt', map_location='cpu')
# print(f"Best checkpoint epoch : {ckpt.get('epoch', 'unknown')}")
# print(f"Best fitness (mAP)    : {ckpt.get('best_fitness', 'unknown'):.4f}")

In [ ]:
def train_model(
    model:      YOLO,
    train_args: dict,
    resume:     bool = True,
) -> object:
    """Train the YOLO model and return the results object."""
    print('🚀 Starting training...')
    results = model.train(resume=resume)
    print('✅ Training complete.')
    return results


def find_best_weights(output_dir: Path) -> Path:
    """Locate best.pt from the most recently modified training run."""
    candidates = sorted(
        output_dir.rglob('weights/best.pt'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f'No best.pt found under {output_dir}')
    return candidates[0]

# RESUME_WEIGHTS = Path('/kaggle/input/models/bakwowijunior/yolo-checkpoint/pytorch/default/1/last.pt')
# last_model = YOLO(RESUME_WEIGHTS)

# training_results = train_model(last_model, train_args)

best_weights = find_best_weights(CFG['output_dir'])
print(f'\n🏆 Best weights: {best_weights}')

## 11 · Training Curve Visualisation

In [ ]:
def plot_training_curves(output_dir: Path) -> None:
    """Parse results.csv and plot loss + metric curves."""
    csvs = sorted(output_dir.rglob('results.csv'),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    if not csvs:
        print('⚠ results.csv not found.')
        return
    df = pd.read_csv(csvs[0])
    df.columns = df.columns.str.strip()

    col_map = {
        'train/box_loss'      : 'tr_box',
        'train/cls_loss'      : 'tr_cls',
        'train/dfl_loss'      : 'tr_dfl',
        'val/box_loss'        : 'vl_box',
        'val/cls_loss'        : 'vl_cls',
        'metrics/mAP50(B)'    : 'mAP50',
        'metrics/mAP50-95(B)' : 'mAP50_95',
        'metrics/precision(B)': 'precision',
        'metrics/recall(B)'   : 'recall',
    }
    df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    pairs = [
        ('tr_box',   'vl_box',   'Box Loss'),
        ('tr_cls',   'vl_cls',   'Class Loss'),
        ('tr_dfl',   None,       'DFL Loss (train)'),
        ('mAP50',    'mAP50_95', 'mAP'),
        ('precision','recall',   'Precision / Recall'),
    ]
    for ax, (c1, c2, title) in zip(axes.flatten(), pairs):
        if c1 in df.columns:
            ax.plot(df[c1], label=c1, lw=1.8)
        if c2 and c2 in df.columns:
            ax.plot(df[c2], label=c2, lw=1.8, linestyle='--')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    axes.flatten()[-1].axis('off')
    plt.suptitle('Training Curves', fontsize=14)
    plt.tight_layout()
    plt.show()


plot_training_curves(CFG['output_dir'])

## 12 · Validation & F1-Optimised Confidence Threshold

In [ ]:
def validate_model(
    weights_path: Path,
    yaml_path:    Path,
    cfg:          dict,
) -> object:
    """Run YOLO validation and return metrics object."""
    val_model = YOLO(str(weights_path))
    metrics = val_model.val(
        data    = str(yaml_path),
        imgsz   = cfg['img_size'],
        batch   = cfg['batch_size'],
        conf    = cfg['conf_threshold'],
        iou     = cfg['iou_threshold'],
        max_det = cfg['max_det'],
        workers = cfg['workers'],
        plots   = True,
        verbose = True,
    )
    return metrics


def print_val_table(
    metrics,
    class_names: list[str],
) -> None:
    """Pretty-print per-class detection metrics."""
    try:
        box = metrics.box
        rows = []
        for i, name in enumerate(class_names):
            rows.append(dict(
                class_name = name,
                precision  = round(float(box.p[i]),    4),
                recall     = round(float(box.r[i]),    4),
                ap50       = round(float(box.ap50[i]), 4),
                ap50_95    = round(float(box.ap[i]),   4),
            ))
        df = pd.DataFrame(rows)
        df.loc[len(df)] = dict(
            class_name='ALL',
            precision=round(float(box.mp),    4),
            recall   =round(float(box.mr),    4),
            ap50     =round(float(box.map50), 4),
            ap50_95  =round(float(box.map),   4),
        )
        print(df.to_string(index=False))
    except Exception as e:
        print(f'Could not extract per-class metrics: {e}')

# best_weights = find_best_weights(CFG['output_dir'])
# print(f'\n🏆 Best weights: {best_weights}')

val_metrics = validate_model(best_weights, yaml_path, CFG)
print_val_table(val_metrics, CFG['class_names'])

In [ ]:
def sweep_confidence_threshold(
    weights_path:  Path,
    val_image_dir: Path,
    val_label_dir: Path,
    cfg:           dict,
    conf_lo:       float = 0.05,
    conf_hi:       float = 0.90,
    conf_step:     float = 0.025,
    min_recall:    float = 0.99,
) -> tuple[float, pd.DataFrame]:
    """
    Sweep confidence thresholds on the val set using IoU-matched TP/FP/FN.
    Returns (best_conf, results_df).

    min_recall: same safety constraint as the classification model —
    any threshold that causes recall < min_recall is penalised (F1→0).
    This prevents the sweep from optimising purely for precision.
    """
    sweep_model = YOLO(str(weights_path))
    img_files   = list(val_image_dir.glob('*.jpg'))
    if not img_files:
        print('⚠ No val images found.')
        return cfg['conf_threshold'], pd.DataFrame()

    # Ground truth: {stem: [(cls, cx, cy, w, h), ...]}
    gt: dict[str, list] = {}
    for lbl in val_label_dir.glob('*.txt'):
        gt[lbl.stem] = [
            tuple(map(float, l.split()))
            for l in lbl.read_text().strip().splitlines() if l.strip()
        ]

    confs   = np.arange(conf_lo, conf_hi + 1e-9, conf_step)
    records = []

    for conf in tqdm(confs, desc='Threshold sweep'):
        tp = fp = fn = 0

        for img_path in img_files:
            preds = sweep_model.predict(
                str(img_path),
                conf    = float(conf),
                iou     = cfg['iou_threshold'],
                verbose = False,
            )[0].boxes

            gts    = gt.get(img_path.stem, [])
            n_gt   = len(gts)
            n_pred = len(preds) if preds is not None else 0

            matched_gt, matched_pr = set(), set()
            if preds is not None and n_gt > 0:
                for pi, box in enumerate(preds.xywhn.cpu().numpy()):
                    px1, py1 = box[0]-box[2]/2, box[1]-box[3]/2
                    px2, py2 = px1+box[2],       py1+box[3]
                    pred_cls  = int(preds.cls[pi])
                    for gi, gt_ann in enumerate(gts):
                        if gi in matched_gt: continue
                        gx1, gy1 = gt_ann[1]-gt_ann[3]/2, gt_ann[2]-gt_ann[4]/2
                        gx2, gy2 = gx1+gt_ann[3], gy1+gt_ann[4]
                        ix = max(0, min(px2,gx2)-max(px1,gx1))
                        iy = max(0, min(py2,gy2)-max(py1,gy1))
                        inter = ix * iy
                        union = box[2]*box[3] + gt_ann[3]*gt_ann[4] - inter
                        iou   = inter / (union + 1e-7)
                        if iou >= 0.5 and pred_cls == int(gt_ann[0]):
                            matched_gt.add(gi)
                            matched_pr.add(pi)
                            break
            tp += len(matched_pr)
            fp += n_pred - len(matched_pr)
            fn += n_gt   - len(matched_gt)

        prec = tp / (tp + fp + 1e-7)
        rec  = tp / (tp + fn + 1e-7)
        f1   = 2 * prec * rec / (prec + rec + 1e-7)

        # Penalise threshold if recall < safety floor
        f1_safe = f1 if rec >= min_recall else 0.0

        records.append(dict(
            conf=round(float(conf), 4),
            precision=round(prec, 4),
            recall   =round(rec,  4),
            f1       =round(f1,   4),
            f1_safe  =round(f1_safe, 4),
            tp=tp, fp=fp, fn=fn,
        ))

    df_sweep = pd.DataFrame(records)
    best_row  = df_sweep.loc[df_sweep['f1_safe'].idxmax()]
    best_conf = float(best_row['conf'])

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    axes[0].plot(df_sweep['conf'], df_sweep['f1'],        label='F1',        lw=2)
    axes[0].plot(df_sweep['conf'], df_sweep['precision'], label='Precision',  lw=1.5, ls='--')
    axes[0].plot(df_sweep['conf'], df_sweep['recall'],    label='Recall',     lw=1.5, ls=':')
    axes[0].axvline(best_conf, color='red', ls='-.', label=f'Best τ={best_conf:.3f}')
    axes[0].axhline(min_recall, color='orange', ls=':', alpha=0.7, label=f'min_recall={min_recall}')
    axes[0].set_xlabel('Confidence Threshold')
    axes[0].set_title('F1 / Precision / Recall vs. Threshold')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(df_sweep['conf'], df_sweep['fp'], label='False Positives', color='firebrick', lw=2)
    axes[1].plot(df_sweep['conf'], df_sweep['fn'], label='False Negatives', color='darkorange', lw=2)
    axes[1].axvline(best_conf, color='red', ls='-.')
    axes[1].set_xlabel('Confidence Threshold')
    axes[1].set_title('FP & FN vs. Threshold')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle('Confidence Threshold Sweep', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f'\n🎯 Optimal conf threshold : {best_conf:.4f}')
    print(f'   F1 (safety-constrained) : {best_row["f1_safe"]:.4f}')
    print(f'   Precision               : {best_row["precision"]:.4f}')
    print(f'   Recall                  : {best_row["recall"]:.4f}')
    print(f'   TP={best_row["tp"]}  FP={best_row["fp"]}  FN={best_row["fn"]}')
    return best_conf, df_sweep


optimal_conf, sweep_df = sweep_confidence_threshold(
    weights_path  = best_weights,
    val_image_dir = CFG['dataset_root'] / 'val' / 'images',
    val_label_dir = CFG['dataset_root'] / 'val' / 'labels',
    cfg           = CFG,
    min_recall    = 0.99,
)
CFG['conf_threshold'] = optimal_conf
print(f'\nCFG["conf_threshold"] updated to {optimal_conf:.4f}')

In [29]:
# import shutil
# import os
# os.remove('/kaggle/working/output.zip')

In [ ]:
def evaluate_binary_f1(
    weights_path:  Path,
    val_image_dir: Path,
    val_label_dir: Path,
    cfg:           dict,
    conf_thresholds: list[float] | None = None,
) -> tuple[float, float, pd.DataFrame]:
    """
    Evaluate **binary** bottle-level F1 (GOOD vs FAULTY) rather than
    per-defect-class mAP.  This is the competition metric.

    Decision rule applied here mirrors the full inference engine:
      - Any always-FAULTY detection  →  bottle = FAULTY
      - Any conditional detection with area > threshold  →  bottle = FAULTY
      - Otherwise  →  bottle = GOOD

    Ground truth for each image is derived from its label file:
      - If label file is empty or absent  →  GOOD  (no defects annotated)
      - If label file has any annotation  →  FAULTY (at least one defect present)
        Note: we use presence of any annotation as ground truth FAULTY here
        because the COCO → YOLO conversion already filtered out GOOD-only labels.

    Returns (best_conf_threshold, best_binary_f1, sweep_df).
    """
    eval_model  = YOLO(str(weights_path))
    img_files   = list(val_image_dir.glob('*.jpg'))
    if not img_files:
        print('⚠ No val images for binary F1 evaluation.')
        return cfg['conf_threshold'], 0.0, pd.DataFrame()

    # Ground truth: any label file with content → FAULTY (1), empty → GOOD (0)
    gt_binary: dict[str, int] = {}
    for img_path in img_files:
        lbl = val_label_dir / f'{img_path.stem}.txt'
        gt_binary[img_path.stem] = (
            1 if (lbl.exists() and lbl.read_text().strip()) else 0
        )

    n_gt_faulty = sum(gt_binary.values())
    n_gt_good   = len(gt_binary) - n_gt_faulty
    print(f'  Val set: {len(img_files)} images | GOOD={n_gt_good} | FAULTY={n_gt_faulty}')

    always_faulty_idx = set(range(16))          # class indices 0–15
    conditional_idx   = set(range(16, 22))      # class indices 16–21

    if conf_thresholds is None:
        conf_thresholds = list(np.arange(0.05, 0.90, 0.025))

    records = []
    for conf in tqdm(conf_thresholds, desc='Binary-F1 sweep'):
        tp = fp = fn = tn = 0

        for img_path in img_files:
            preds  = eval_model.predict(
                str(img_path),
                conf=float(conf), iou=cfg['iou_threshold'],
                verbose=False,
            )[0].boxes

            gt_label = gt_binary[img_path.stem]

            # Apply defect logic to get predicted binary label
            pred_faulty = False
            if preds is not None and len(preds) > 0:
                img_cv    = cv2.imread(str(img_path))
                ih, iw    = img_cv.shape[:2]
                img_area  = ih * iw

                for i in range(len(preds)):
                    cls   = int(preds.cls[i])
                    xyxy  = preds.xyxy[i].cpu().numpy()
                    area  = (xyxy[2]-xyxy[0]) * (xyxy[3]-xyxy[1])

                    if cls in always_faulty_idx:
                        pred_faulty = True
                        break
                    elif cls in conditional_idx:
                        cls_name = cfg['class_names'][cls]
                        thresh   = cfg['area_thresholds'].get(cls_name, 0)
                        if area > thresh:
                            pred_faulty = True
                            break

            pred_label = int(pred_faulty)

            if gt_label == 1 and pred_label == 1: tp += 1
            elif gt_label == 0 and pred_label == 1: fp += 1
            elif gt_label == 1 and pred_label == 0: fn += 1
            else: tn += 1

        prec   = tp / (tp + fp + 1e-7)
        rec    = tp / (tp + fn + 1e-7)
        f1     = 2 * prec * rec / (prec + rec + 1e-7)
        acc    = (tp + tn) / (tp + tn + fp + fn + 1e-7)

        records.append(dict(
            conf=round(float(conf),4), precision=round(prec,4),
            recall=round(rec,4), f1=round(f1,4), accuracy=round(acc,4),
            tp=tp, fp=fp, fn=fn, tn=tn,
        ))

    df = pd.DataFrame(records)
    best = df.loc[df['f1'].idxmax()]

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    axes[0].plot(df['conf'], df['f1'],        label='Binary F1',  lw=2.5, color='steelblue')
    axes[0].plot(df['conf'], df['precision'], label='Precision',  lw=1.5, ls='--', color='coral')
    axes[0].plot(df['conf'], df['recall'],    label='Recall',     lw=1.5, ls=':', color='green')
    axes[0].axvline(best['conf'], color='red', ls='-.', lw=1.5,
                    label=f"Best τ={best['conf']:.3f} → F1={best['f1']:.4f}")
    axes[0].axhline(0.98, color='gold', ls=':', alpha=0.8, label='F1=0.98 target')
    axes[0].set_xlabel('Confidence Threshold'); axes[0].set_title('Binary Bottle-Level F1')
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    axes[1].stackplot(df['conf'],
                      df['tp'], df['tn'], df['fp'], df['fn'],
                      labels=['TP','TN','FP','FN'],
                      colors=['#2ecc71','#3498db','#e74c3c','#e67e22'],
                      alpha=0.75)
    axes[1].axvline(best['conf'], color='red', ls='-.')
    axes[1].set_xlabel('Confidence Threshold'); axes[1].set_title('Confusion counts vs Threshold')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.suptitle('Binary Bottle-Level Evaluation', fontsize=12)
    plt.tight_layout(); plt.show()

    print(f'\n── Binary Bottle-Level Results ──────────────────────────────────────')
    print(f'  Best confidence threshold : {best["conf"]:.4f}')
    print(f'  Binary F1 (FAULTY class)  : {best["f1"]:.4f}   ← competition metric')
    print(f'  Precision                 : {best["precision"]:.4f}')
    print(f'  Recall                    : {best["recall"]:.4f}')
    print(f'  Accuracy                  : {best["accuracy"]:.4f}')
    print(f'  TP={int(best["tp"])}  TN={int(best["tn"])}  FP={int(best["fp"])}  FN={int(best["fn"])}')
    print(f'  F1 ≥ 0.98 target          : {"✓ MET" if best["f1"] >= 0.98 else "✗ NOT YET"}')
    print(f'──────────────────────────────────────────────────────────────────────')
    return float(best['conf']), float(best['f1']), df


# ── Run binary F1 evaluation on val set ──────────────────────────────────────
print('Running binary bottle-level F1 evaluation (competition metric)...')
best_binary_conf, best_binary_f1, binary_sweep_df = evaluate_binary_f1(
    weights_path  = best_weights,
    val_image_dir = CFG['dataset_root'] / 'val' / 'images',
    val_label_dir = CFG['dataset_root'] / 'val' / 'labels',
    cfg           = CFG,
)
# Use this threshold for submission — it optimises the actual competition metric
CFG['conf_threshold'] = best_binary_conf
print(f'\nCFG["conf_threshold"] updated to {best_binary_conf:.4f}  (optimised for binary F1)')


In [ ]:
def analyse_hard_cases(
    weights_path:  Path,
    val_image_dir: Path,
    val_label_dir: Path,
    cfg:           dict,
    top_n:         int = 20,
) -> pd.DataFrame:
    """
    Identify the hardest bottles — FAULTY images the model misses (FN)
    and GOOD images it falsely rejects (FP) — at the best threshold.
    Prints the defect type breakdown for false negatives.

    Knowing WHICH defect types cause false negatives tells us exactly
    which classes need more oversampling in the next training run.
    Lesson from classification model: contamination_dark + water_drop
    co-occurrence was the dominant FN pattern after 3 training runs.
    """
    eval_model = YOLO(str(weights_path))
    img_files  = list(val_image_dir.glob('*.jpg'))
    if not img_files:
        print('⚠ No val images.')
        return pd.DataFrame()

    always_faulty_idx = set(range(16))
    conditional_idx   = set(range(16, 22))
    conf = cfg['conf_threshold']

    rows = []
    for img_path in tqdm(img_files, desc='Hard-case analysis'):
        lbl = val_label_dir / f'{img_path.stem}.txt'
        gt_lines = lbl.read_text().strip().splitlines() if lbl.exists() else []
        gt_label = 1 if gt_lines else 0
        gt_classes = sorted({cfg['class_names'][int(float(l.split()[0]))]
                             for l in gt_lines if l.strip()})

        preds = eval_model.predict(
            str(img_path), conf=float(conf),
            iou=cfg['iou_threshold'], verbose=False,
        )[0].boxes

        pred_faulty = False
        pred_classes = []
        if preds is not None and len(preds) > 0:
            img_cv = cv2.imread(str(img_path))
            for i in range(len(preds)):
                cls  = int(preds.cls[i])
                name = cfg['class_names'][cls]
                xyxy = preds.xyxy[i].cpu().numpy()
                area = (xyxy[2]-xyxy[0]) * (xyxy[3]-xyxy[1])
                if cls in always_faulty_idx:
                    pred_faulty = True
                    pred_classes.append(name)
                elif cls in conditional_idx:
                    if area > cfg['area_thresholds'].get(name, 0):
                        pred_faulty = True
                        pred_classes.append(name)

        pred_label = int(pred_faulty)
        outcome = ('TP' if gt_label==1 and pred_label==1 else
                   'TN' if gt_label==0 and pred_label==0 else
                   'FP' if gt_label==0 and pred_label==1 else 'FN')

        rows.append(dict(
            stem        = img_path.stem,
            gt_label    = gt_label,
            pred_label  = pred_label,
            outcome     = outcome,
            gt_classes  = ', '.join(gt_classes),
            pred_classes= ', '.join(pred_classes),
        ))

    df = pd.DataFrame(rows)

    print('\n── False Negatives (FAULTY missed by model) ─────────────────────────')
    fn_df = df[df['outcome'] == 'FN']
    print(f'  Total FN: {len(fn_df)}')
    if not fn_df.empty:
        all_fn_classes = pd.Series(
            [c for row in fn_df['gt_classes'] for c in row.split(', ') if c]
        ).value_counts()
        print('  Defect types causing false negatives (→ need more oversampling):')
        for cls, cnt in all_fn_classes.items():
            print(f'    {cls:30s}: {cnt}')

    print('\n── False Positives (GOOD rejected by model) ─────────────────────────')
    fp_df = df[df['outcome'] == 'FP']
    print(f'  Total FP: {len(fp_df)}')
    if not fp_df.empty:
        print(f'  Model detected: {fp_df["pred_classes"].value_counts().head(10).to_string()}')

    return df


hard_case_df = analyse_hard_cases(
    weights_path  = best_weights,
    val_image_dir = CFG['dataset_root'] / 'val' / 'images',
    val_label_dir = CFG['dataset_root'] / 'val' / 'labels',
    cfg           = CFG,
)


## 13 · Export & Optimisation

In [ ]:
def export_model(
    weights_path:   Path,
    export_formats: list[str],
    img_size:       int,
    half:           bool,
    export_dir:     Path,
) -> dict[str, Path]:
    """Export best.pt to each requested format and copy to export_dir."""
    export_dir.mkdir(parents=True, exist_ok=True)
    export_obj = YOLO(str(weights_path))
    exported: dict[str, Path] = {}

    for fmt in export_formats:
        print(f'\n🔧 Exporting → {fmt.upper()} ...')
        try:
            out = export_obj.export(
                format   = fmt,
                imgsz    = img_size,
                half     = half,
                dynamic  = False,
                simplify = (fmt == 'onnx'),
                opset    = 17 if fmt == 'onnx' else None,
            )
            dst = export_dir / Path(out).name
            shutil.copy2(out, dst)
            exported[fmt] = dst
            print(f'  ✅ Saved to {dst}')
        except Exception as e:
            print(f'  ⚠ {fmt} failed: {e}')

    return exported


def benchmark_speed(
    weights_path: Path,
    img_size:     int,
    n_warmup:     int = 10,
    n_runs:       int = 100,
) -> dict:
    """Measure inference latency on a synthetic image."""
    m = YOLO(str(weights_path))
    dummy = np.random.randint(0, 255, (img_size, img_size, 3), np.uint8)
    for _ in range(n_warmup):
        m.predict(dummy, verbose=False)
    times = []
    for _ in range(n_runs):
        t = time.perf_counter()
        m.predict(dummy, verbose=False)
        times.append((time.perf_counter() - t) * 1000)
    t = np.array(times)
    stats = dict(mean_ms=round(t.mean(),2), std_ms=round(t.std(),2),
                 p95_ms=round(np.percentile(t,95),2), fps=round(1000/t.mean(),1))
    print(f'⚡ Latency: {stats["mean_ms"]} ± {stats["std_ms"]} ms  '
          f'P95={stats["p95_ms"]} ms  FPS={stats["fps"]}')
    return stats


exported = export_model(
    weights_path   = best_weights,
    export_formats = CFG['export_formats'],
    img_size       = CFG['img_size'],
    half           = CFG['half'],
    export_dir     = CFG['model_export_dir'],
)
speed = benchmark_speed(best_weights, CFG['img_size'])

## 14 · Inference Engine — Conditional Area-Gate Defect Logic

### Decision rules (from domain specification)

```
For each detected bbox:
  class_name ∈ always_faulty  → DEFECT = True   (regardless of area)
  class_name ∈ conditional    → DEFECT = True   iff  bbox_area_px² > area_threshold[class]

Image verdict:
  FAULTY  if ≥1 confirmed DEFECT in the image
  GOOD    otherwise
```


In [ ]:
@dataclass
class Detection:
    class_idx    : int
    class_name   : str
    confidence   : float
    bbox_xyxy    : tuple[float, float, float, float]   # absolute pixels in crop
    bbox_area_px : float    # w × h in pixels (crop coordinates)
    area_thresh  : float    # 0 for always-faulty; threshold for conditionals
    is_confirmed : bool     # True if area > threshold (or always-faulty)


@dataclass
class InspectionResult:
    image_path        : str
    image_hw          : tuple[int, int]
    all_detections    : list[Detection] = field(default_factory=list)
    confirmed_defects : list[Detection] = field(default_factory=list)
    is_defective      : bool = False
    defect_classes    : list[str] = field(default_factory=list)
    summary           : str = ''


# Classes that are always a defect (no area gate)
_ALWAYS_FAULTY = set(CFG['class_names'][:16])
# Conditional classes
_CONDITIONAL   = set(CFG['class_names'][16:])


def apply_defect_logic(
    boxes,                       # ultralytics Boxes from model.predict()
    img_h:          int,
    img_w:          int,
    class_names:    list[str],
    area_thresholds: dict[str, float],
    conf_threshold: float,
) -> list[Detection]:
    """
    Convert raw YOLO detections to Detection objects with the area gate applied.

    Area gate:
      - Always-FAULTY classes: is_confirmed = True always
      - Conditional classes  : is_confirmed = (bbox_area_px > threshold)

    bbox_area_px is computed in crop-space (absolute pixels), which matches
    the original pixel area because the crop is a 1:1 cut of the raw image
    (no resize yet). YOLO internal resize to 640×640 is reversed automatically
    by Ultralytics — xyxy in model output are in original-crop-image space.
    """
    detections = []
    if boxes is None or len(boxes) == 0:
        return detections

    xyxy_all = boxes.xyxy.cpu().numpy()
    conf_all = boxes.conf.cpu().numpy()
    cls_all  = boxes.cls.cpu().numpy().astype(int)

    cls_conf_threshold = cfg.get('per_class_conf', {}).get(cls_name, conf_threshold)
    
    for xyxy, conf, cls in zip(xyxy_all, conf_all, cls_all):
        if conf < conf_threshold:
            continue
        if conf < cls_conf_threshold:
            continue

        x1, y1, x2, y2 = xyxy
        bw   = max(0.0, x2 - x1)
        bh   = max(0.0, y2 - y1)
        area = bw * bh

        cls_name = class_names[cls] if cls < len(class_names) else f'cls_{cls}'

        if cls_name in _ALWAYS_FAULTY:
            thresh      = 0.0
            is_confirmed = True
        elif cls_name in _CONDITIONAL:
            thresh       = float(area_thresholds.get(cls_name, 0.0))
            is_confirmed = (area > thresh)
        else:
            # Unknown / GOOD class leaked in — skip
            continue

        detections.append(Detection(
            class_idx    = int(cls),
            class_name   = cls_name,
            confidence   = float(conf),
            bbox_xyxy    = (float(x1), float(y1), float(x2), float(y2)),
            bbox_area_px = float(area),
            area_thresh  = thresh,
            is_confirmed = is_confirmed,
        ))

    return detections


def extract_roi_for_inference(
    image_path: str | Path,
    roi_map:    dict[str, tuple[int, int, int]],
    cfg:        dict,
) -> tuple[np.ndarray, int, int]:
    """
    Crop to bottle-base ROI for inference.
    Returns (cropped_bgr_image, x_offset, y_offset).
    """
    img  = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f'Cannot load: {image_path}')

    fname = Path(image_path).name
    if fname in roi_map:
        cx, cy, r = roi_map[fname]
    else:
        h, w = img.shape[:2]
        cx, cy, r = w // 2, h // 2, min(w, h) // 2 - 20

    cropped, x_off, y_off, _, _ = crop_to_roi(img, cx, cy, r)
    return cropped, x_off, y_off


def inspect_image(
    image_path:  str | Path,
    model:       YOLO,
    cfg:         dict,
    roi_map:     dict[str, tuple[int, int, int]],
) -> InspectionResult:
    """
    Full inspection pipeline for a single image.

    Steps:
      1. Crop to bottle-base ROI
      2. YOLO inference on the crop
      3. Area-gate defect logic
      4. Bottle verdict (GOOD / FAULTY)
    """
    cropped, x_off, y_off = extract_roi_for_inference(image_path, roi_map, cfg)
    h, w = cropped.shape[:2]

    preds = model.predict(
        cropped,
        imgsz   = cfg['img_size'],
        conf    = cfg['conf_threshold'],
        iou     = cfg['iou_threshold'],
        max_det = cfg['max_det'],
        verbose = False,
    )[0]

    all_dets = apply_defect_logic(
        preds.boxes,
        img_h           = h,
        img_w           = w,
        class_names     = cfg['class_names'],
        area_thresholds = cfg['area_thresholds'],
        conf_threshold  = cfg['conf_threshold'],
    )

    confirmed   = [d for d in all_dets if d.is_confirmed]
    defect_cls  = sorted({d.class_name for d in confirmed})
    is_defective = len(confirmed) > 0

    lines = [f"{'⚠ DEFECTIVE' if is_defective else '✅ GOOD'} — {Path(image_path).name}"]
    for d in all_dets:
        area_info = (
            f'area={d.bbox_area_px:.0f}px² (thresh={d.area_thresh:.0f})'
            if d.area_thresh > 0 else 'always-faulty'
        )
        gate = '✓ DEFECT' if d.is_confirmed else '~ sub-threshold'
        lines.append(
            f'  {d.class_name:25s} conf={d.confidence:.3f}  {area_info:35s}  [{gate}]'
        )

    return InspectionResult(
        image_path        = str(image_path),
        image_hw          = (h, w),
        all_detections    = all_dets,
        confirmed_defects = confirmed,
        is_defective      = is_defective,
        defect_classes    = defect_cls,
        summary           = '\n'.join(lines),
    )


print('✅ Defect inspection engine ready.')

## 15 · Batch Inspection, Visualisation & Report

In [ ]:
# Colour palette: red = confirmed defect, green = sub-threshold, grey = always-faulty but low area
_COL_DEFECT     = (220, 30,  30)
_COL_SUBTHRESH  = (30,  180, 30)
_COL_BANNER_BAD = (180, 20,  20)
_COL_BANNER_OK  = (20,  140, 20)


def draw_inspection(
    image_path: str | Path,
    result:     InspectionResult,
    roi_map:    dict[str, tuple[int, int, int]],
    cfg:        dict,
) -> np.ndarray:
    """Return an annotated BGR image of the ROI crop."""
    cropped, *_ = extract_roi_for_inference(image_path, roi_map, cfg)
    vis = cropped.copy()

    for d in result.all_detections:
        x1, y1, x2, y2 = map(int, d.bbox_xyxy)
        color  = _COL_DEFECT if d.is_confirmed else _COL_SUBTHRESH
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

        # Label
        area_txt = f' {d.bbox_area_px:.0f}px²' if d.area_thresh > 0 else ''
        label = f'{d.class_name} {d.confidence:.2f}{area_txt}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(vis, (x1, max(0, y1-th-6)), (x1+tw+2, y1), color, -1)
        cv2.putText(vis, label, (x1+1, y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1, cv2.LINE_AA)

    # Banner
    banner_col  = _COL_BANNER_BAD if result.is_defective else _COL_BANNER_OK
    banner_text = (f'DEFECT: {", ".join(result.defect_classes)}'
                   if result.is_defective else 'PASS')
    cv2.rectangle(vis, (0, 0), (vis.shape[1], 34), banner_col, -1)
    cv2.putText(vis, banner_text, (8, 23),
                cv2.FONT_HERSHEY_DUPLEX, 0.7, (255,255,255), 1, cv2.LINE_AA)

    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)


def batch_inspect(
    image_paths: list[Path],
    model:       YOLO,
    cfg:         dict,
    roi_map:     dict,
    max_display: int = 6,
) -> list[InspectionResult]:
    """Inspect a list of images and display an annotated grid."""
    results     = []
    show_imgs   = []
    show_titles = []

    for path in tqdm(image_paths, desc='Inspecting'):
        try:
            r = inspect_image(path, model, cfg, roi_map)
        except Exception as e:
            print(f'  ⚠ {Path(path).name}: {e}')
            continue
        print(r.summary)
        results.append(r)

        if len(show_imgs) < max_display:
            show_imgs.append(draw_inspection(path, r, roi_map, cfg))
            verdict = 'DEFECT' if r.is_defective else 'PASS'
            show_titles.append(f'{Path(path).name}\n[{verdict}]')

    n    = len(show_imgs)
    cols = min(n, 3)
    rows = math.ceil(n / cols) if cols else 1
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
    axes_flat  = np.array(axes).flatten() if n > 1 else [axes]

    for ax, img, title in zip(axes_flat, show_imgs, show_titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=9,
                     color='red' if 'DEFECT' in title else 'green')
        ax.axis('off')
    for ax in axes_flat[n:]:
        ax.axis('off')

    plt.suptitle('Bottle Inspection Results', fontsize=13)
    plt.tight_layout()
    plt.show()
    return results


def build_inspection_report(
    results:     list[InspectionResult],
    class_names: list[str],
    out_path:    Optional[Path] = None,
) -> pd.DataFrame:
    """Create a per-image summary DataFrame and optionally save as CSV."""
    rows = []
    for r in results:
        row = {
            'image'          : Path(r.image_path).name,
            'is_defective'   : int(r.is_defective),
            'n_all_det'      : len(r.all_detections),
            'n_confirmed'    : len(r.confirmed_defects),
            'defect_classes' : ', '.join(r.defect_classes) or '-',
        }
        for cn in class_names:
            row[f'n_{cn}'] = sum(1 for d in r.confirmed_defects
                                  if d.class_name == cn)
        rows.append(row)

    df = pd.DataFrame(rows)
    if out_path:
        df.to_csv(out_path, index=False)
        print(f'  Report saved → {out_path}')

    print('\n── Inspection Summary ────────────────────────────────')
    print(f'  Total images  : {len(df)}')
    print(f'  Defective     : {df["is_defective"].sum()}')
    print(f'  Clean         : {(df["is_defective"]==0).sum()}')
    if not df.empty:
        breakdown = pd.Series(
            [cls for r in results for cls in r.defect_classes]
        ).value_counts()
        if not breakdown.empty:
            print('\n  Defect type breakdown:')
            for cls, cnt in breakdown.items():
                print(f'    {cls:30s}: {cnt}')
    return df


# ── Run on test / val images ─────────────────────────────────────────────────
inference_model = YOLO(str(best_weights))

test_imgs = list((CFG['dataset_root'] / 'test' / 'images').glob('*.jpg'))
if not test_imgs:
    test_imgs = list((CFG['dataset_root'] / 'val' / 'images').glob('*.jpg'))

if test_imgs:
    all_results = batch_inspect(
        image_paths = test_imgs[:50],   # first 50 for speed; remove slice for full run
        model       = inference_model,
        cfg         = CFG,
        roi_map     = roi_map,
    )
    report_df = build_inspection_report(
        all_results,
        CFG['class_names'],
        out_path = CFG['predictions_csv'],
    )
else:
    print('⚠ No test/val images found.')

## 16 · Submission CSV Generation

In [ ]:
def load_test_roi_map(
    test_json_path: Path,
    roi_category_id: int = 22,
) -> dict[str, tuple[int, int, int]]:
    """
    Build ROI map from the test annotations JSON.
    Same COCO format as training annotations.
    Falls back to empty dict (fixed-fallback ROI used in inspect_image)
    if the file is missing or has no annotations.
    """
    if not test_json_path.exists():
        print(f'  ⚠ {test_json_path} not found — using fixed-fallback ROI.')
        return {}
    coco_t, id_to_fn_t, _ = load_coco(test_json_path)
    return build_roi_map(coco_t, id_to_fn_t, roi_category_id)


def make_submission(
    test_images_dir:      Path,
    best_weights:         Path,
    cfg:                  dict,
    test_roi_map:         dict,
    submission_csv:       Path,
    calibrated_threshold: Optional[float] = None,
) -> pd.DataFrame:
    """
    Run the full inference + defect-gate pipeline on every test image
    and write submission.csv with columns [image_id, target].

    target = 1 (FAULTY) or 0 (GOOD).
    """
    if calibrated_threshold is not None:
        cfg = {**cfg, 'conf_threshold': calibrated_threshold}
        print(f'  Using calibrated threshold: {calibrated_threshold:.4f}')

    sub_model    = YOLO(str(best_weights))
    image_paths  = sorted(test_images_dir.glob('*.png')) + \
                   sorted(test_images_dir.glob('*.jpg'))

    if not image_paths:
        raise FileNotFoundError(f'No images found in {test_images_dir}')
    print(f'  {len(image_paths)} test images found.')

    n_with_roi = sum(1 for p in image_paths if p.name in test_roi_map)
    print(f'  ROI from COCO: {n_with_roi}  |  Fixed fallback: {len(image_paths)-n_with_roi}')

    rows = []
    for path in tqdm(image_paths, desc='Submission inference'):
        try:
            result = inspect_image(path, sub_model, cfg, test_roi_map)
            target = int(result.is_defective)
        except Exception as e:
            print(f'    ⚠ {path.name}: {e} — defaulting to 0 (GOOD)')
            target = 0
        rows.append({'image_id': path.name, 'target': target})

    df_sub = pd.DataFrame(rows)
    df_sub.to_csv(submission_csv, index=False)

    n_f = (df_sub['target'] == 1).sum()
    n_g = (df_sub['target'] == 0).sum()
    print(f'\n  Submission saved → {submission_csv}')
    print(f'  GOOD  : {n_g} ({100*n_g/len(df_sub):.1f}%)')
    print(f'  FAULTY: {n_f} ({100*n_f/len(df_sub):.1f}%)')
    print(f'  Total : {len(df_sub)}')
    return df_sub


# ── Load test ROI annotations ─────────────────────────────────────────────────
print('Loading test ROI annotations...')
test_roi_map = load_test_roi_map(CFG['coco_json_test'], CFG['roi_category_id'])

# ── Generate submission ───────────────────────────────────────────────────────
# Pass the threshold calibrated in Section 12 (stored in CFG['conf_threshold'])
submission_df = make_submission(
    test_images_dir      = CFG['test_images_dir'],
    best_weights         = best_weights,
    cfg                  = CFG,
    test_roi_map         = test_roi_map,
    submission_csv       = CFG['submission_csv'],
    calibrated_threshold = CFG['conf_threshold'],
)
print(submission_df.head(10))

---
## Summary

| Component | Detail |
|-----------|--------|
| **Model** | YOLO11-nano (auto-fallback: n → s → YOLOv8n) |
| **ROI** | COCO annotation category_id=22 → circle crop; fixed fallback |
| **Classes** | 16 always-FAULTY + 6 conditional = 22 detection targets |
| **GOOD labels** | Water drop, foam residue, embossing, no fault — excluded from YOLO targets |
| **Area gate** | Conditional defects only confirmed as FAULTY if `bbox_area_px² > threshold` |
| **Offline aug** | Minority-class oversampling via Albumentations (tight colour jitter; no CoarseDropout) |
| **Online aug** | Mosaic, MixUp, CopyPaste, HSV, affine, flip — YOLO native |
| **Threshold** | F1-optimised confidence sweep with min_recall=0.99 safety constraint |
| **Export** | ONNX FP16 |
| **Output** | `InspectionResult` dataclass + annotated images + predictions.csv + submission.csv |

### Hard cases noted from classification model analysis
Images containing **contamination dark + water drop + scuffing** co-occurring consistently  
score lower than pure-contamination images. The detection model mitigates this by predicting  
separate bboxes for each defect type — the `contamination_dark` bbox triggers the always-FAULTY  
rule regardless of what other GOOD-signal annotations surround it.
